# 4. Bayes by Backprop

The one built from the mathematics rather than from a recipe. Every weight becomes
a Gaussian: instead of one number per weight you learn a mean and a standard
deviation, so the network is a distribution over networks and a forward pass draws
one of them.

You write four things, each checked separately:

1. `BayesLinear.__init__`, which registers the mean and log-standard-deviation;
2. `BayesLinear.forward`, which samples with the reparameterization trick;
3. `BayesLinear.kl_divergence`, the closed-form Gaussian KL;
4. `elbo_loss`, where the KL is scaled by `1 / n_train`.

About 3 hours. This is the longest notebook and the only one where a wrong answer
looks exactly like a right one, so run the check cells as you go.

## 4.1 The objective

We want the posterior `p(W | D)`, which is intractable, so we pick a family
`q(W)` and find the member closest to it. For any `q`,

    log p(D) >= E_q[ log p(D | W) ]  -  KL( q(W) || p(W) )  =:  L(q)
               \___________________/    \________________/
                 fit the data             stay near the prior

and the gap is exactly `KL(q(W) || p(W | D))`. So maximising `L` minimises a
divergence to the true posterior, which is what makes this an approximation with a
target rather than a heuristic.

The family is mean-field Gaussian: every weight independent,

    q(w_j) = N(mu_j, sigma_j^2),     p(w_j) = N(0, prior_std^2)

Independence is the strong assumption. It cannot represent correlations between
weights, and because the divergence being minimised is `KL(q || p)` rather than
the other way round, the fitted `q` prefers to sit inside one mode and be too
narrow. The bonus notebook measures exactly how narrow, on a problem where the
right answer is known.

`ASSIGNMENT.md` has the derivations; `THEORY.md` has them in full.

In [ ]:
import math
import sys
import time
from functools import partial

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from bdl.data import load_track, make_toy1d, toy1d_grid, toy1d_noise_std
from bdl.metrics import evaluate, interval_coverage, results_table
from bdl.models import (
    build_model,
    count_parameters,
    default_loss,
    fit,
    gaussian_head,
    set_seed,
    track_hparams,
)
from bdl.plots import plot_aleatoric_recovery, plot_band, plot_reliability, plot_uncertainty_vs_x
from bdl.store import load_run, save_run

%matplotlib inline

ds = make_toy1d()
grid = toy1d_grid()


# The functions you wrote in notebooks 01 and 02, repeated so this notebook stands
# on its own. The last two are only used on Track C.
def calibration_error(mu, sigma, y, n_bins=15):
    levels = torch.linspace(0.0, 1.0, n_bins + 2)[1:-1]
    gaps = [abs(interval_coverage(mu, sigma, y, float(q)) - float(q)) for q in levels]
    return float(np.mean(gaps))


def decompose_variance(mu, sigma):
    aleatoric = (sigma**2).mean(dim=0)
    epistemic = mu.var(dim=0, unbiased=False)
    return aleatoric + epistemic, aleatoric, epistemic


def decompose_entropy(probs):
    eps = 1e-12
    probs = probs.clamp_min(eps)
    mean_p = probs.mean(dim=0)
    total = -(mean_p * torch.log(mean_p.clamp_min(eps))).sum(-1)
    aleatoric = -(probs * torch.log(probs)).sum(-1).mean(dim=0)
    return total, aleatoric, total - aleatoric


def calibration_error_probs(probs, y, n_bins=15):
    p = probs.mean(0)
    conf, hat = p.max(dim=-1)
    correct = (hat == y.reshape(-1)).float()
    edges = torch.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if bool(in_bin.any()):
            ece += float(in_bin.float().mean()) * abs(
                float(correct[in_bin].mean()) - float(conf[in_bin].mean())
            )
    return ece

## 4.2 The layer

Three methods to write.

**`__init__`.** Register four parameters:

    weight_mu         [out_features, in_features]   init U(-b, b), b = 1/sqrt(in_features)
    weight_log_sigma  [out_features, in_features]   filled with init_log_sigma
    bias_mu           [out_features]                zeros
    bias_log_sigma    [out_features]                filled with init_log_sigma

We store `log sigma` rather than `sigma` so the standard deviation is positive for
free and the optimiser sees an unconstrained parameter. `sigma` starts small,
`exp(-5) = 0.0067`, so the network begins almost deterministic and can start
fitting. Start it at 1 and the forward pass is mostly noise, the gradients are
useless, and training never gets going.

**`forward`.** Sample a weight matrix with the reparameterization trick:

    sigma_w = exp(weight_log_sigma)
    eps_w   = torch.randn_like(sigma_w)
    w       = weight_mu + sigma_w * eps_w

then the same for the bias, then `F.linear(x, w, b)`. Draw a fresh `eps` on every
call.

Why not `torch.normal(mu, sigma)`, which draws from the same distribution: the
randomness has to sit *outside* the parameters for the gradient to reach them.
With `mu + sigma * eps` the sampled weight is a differentiable function of `mu`
and `sigma`, and `eps` comes from a fixed distribution. `torch.normal` gives you a
tensor with no gradient path, so `weight_log_sigma.grad` stays `None` and `sigma`
never trains. Nothing crashes; you get an expensive deterministic network. There
is a check for exactly that.

**`kl_divergence`.** For one weight, with prior `N(0, s^2)` and posterior
`N(mu, sigma^2)`:

    KL = log(s / sigma) + (sigma^2 + mu^2) / (2 s^2) - 1/2

Sum it over the weights and the biases. Two sanity checks to remember: it is never
negative, and it is exactly zero when `mu = 0` and `sigma = s`.

In [ ]:
class BayesLinear(nn.Module):
    """A linear layer whose weights are Gaussian random variables.

    A drop-in replacement for nn.Linear: build_model(ds, layer=BayesLinear) gives
    a Bayesian MLP.
    """

    def __init__(self, in_features, out_features, prior_std=1.0, init_log_sigma=-5.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.prior_std = prior_std

        # ---- TODO ------------------------------------------------------------
        # register four nn.Parameters: weight_mu, weight_log_sigma, bias_mu,
        # bias_log_sigma, with the shapes and initial values given above
        raise NotImplementedError
        # ----------------------------------------------------------------------

        # When False, use the posterior means. Handy for a sanity check, but note
        # that the mean network is not the posterior predictive mean.
        self.sample = True

    def forward(self, x):
        if self.sample:
            # ---- TODO --------------------------------------------------------
            # w = weight_mu + exp(weight_log_sigma) * randn_like(...)
            # b = bias_mu   + exp(bias_log_sigma)   * randn_like(...)
            raise NotImplementedError
            # ------------------------------------------------------------------
        else:
            w, b = self.weight_mu, self.bias_mu
        return F.linear(x, w, b)

    def kl_divergence(self):
        """KL(q || p) for this layer, summed over its weights and biases."""
        # ---- TODO ------------------------------------------------------------
        # log(s / sigma) + (sigma^2 + mu^2) / (2 s^2) - 1/2, summed
        raise NotImplementedError
        # ----------------------------------------------------------------------

In [ ]:
# Provided: the total KL over every Bayesian layer in a model, and a switch to
# turn weight sampling off.
def model_kl(model):
    total = torch.zeros(())
    for module in model.modules():
        if isinstance(module, BayesLinear):
            total = total + module.kl_divergence()
    return total


def set_sampling(model, sample):
    for module in model.modules():
        if isinstance(module, BayesLinear):
            module.sample = sample

In [ ]:
# ---- check your work: the layer --------------------------------------------
torch.manual_seed(0)

# 1. Four learnable parameters, with the right shapes.
layer = BayesLinear(3, 2)
names = {n for n, p in layer.named_parameters() if p.requires_grad}
assert names == {"weight_mu", "weight_log_sigma", "bias_mu", "bias_log_sigma"}, names
assert layer.weight_mu.shape == (2, 3) and layer.bias_mu.shape == (2,)

# 2. Sampling makes the output stochastic, and switching it off makes it fixed.
noisy = BayesLinear(4, 3, init_log_sigma=0.0)  # sigma = 1
x = torch.randn(8, 4)
assert not torch.allclose(noisy(x), noisy(x)), "the output should differ between calls"
noisy.sample = False
assert torch.allclose(noisy(x), noisy(x)), "with sample=False the layer is deterministic"

# 3. The whole point of the trick: log_sigma must receive gradient.
layer = BayesLinear(5, 2, init_log_sigma=-1.0)
layer(torch.randn(16, 5)).pow(2).sum().backward()
assert layer.weight_log_sigma.grad is not None, "no gradient reached log_sigma"
assert float(layer.weight_log_sigma.grad.abs().sum()) > 0

# 4. The sampled weights really are N(mu, sigma^2). Feed a one-hot input so the
#    output reads off a single weight, then recover its mean and std from 4000 draws.
layer = BayesLinear(2, 1, init_log_sigma=math.log(0.3))
with torch.no_grad():
    layer.weight_mu.fill_(2.0)
    layer.bias_mu.fill_(0.0)
    layer.bias_log_sigma.fill_(-20.0)
draws = torch.cat([layer(torch.tensor([[1.0, 0.0]])) for _ in range(4000)]).flatten().detach()
assert abs(float(draws.mean()) - 2.0) < 0.03, float(draws.mean())
assert abs(float(draws.std()) - 0.3) < 0.03, float(draws.std())

print("OK   parameters, sampling, gradient path and the sampled distribution")

In [ ]:
# ---- check your work: the KL ------------------------------------------------
torch.manual_seed(3)

# 1. Exactly zero when the posterior is the prior.
layer = BayesLinear(6, 4, prior_std=1.0, init_log_sigma=0.0)  # sigma = 1
with torch.no_grad():
    layer.weight_mu.zero_()
    layer.bias_mu.zero_()
kl_zero = float(layer.kl_divergence().detach())
assert abs(kl_zero) < 1e-5, kl_zero

# 2. Never negative, whatever the width.
for log_sigma in (-4.0, -1.0, 0.0, 1.0):
    kl = float(BayesLinear(5, 3, prior_std=0.5, init_log_sigma=log_sigma).kl_divergence().detach())
    assert kl >= -1e-6, kl

# 3. Against a Monte Carlo estimate of E_q[log q - log p]. This is the check that
#    catches a missing factor of two or a dropped log term: the estimate knows
#    nothing about your algebra.
prior_std = 0.7
layer = BayesLinear(3, 2, prior_std=prior_std, init_log_sigma=-0.8)
with torch.no_grad():
    layer.weight_mu.normal_(0, 0.5)
    layer.bias_mu.normal_(0, 0.5)
analytic = float(layer.kl_divergence().detach())

q_mu = torch.cat([layer.weight_mu.flatten(), layer.bias_mu.flatten()]).detach()
q_sigma = torch.exp(
    torch.cat([layer.weight_log_sigma.flatten(), layer.bias_log_sigma.flatten()])
).detach()
q = torch.distributions.Normal(q_mu, q_sigma)
p = torch.distributions.Normal(torch.zeros_like(q_mu), torch.full_like(q_mu, prior_std))
w = q.sample((200_000,))
mc = float((q.log_prob(w).sum(-1) - p.log_prob(w).sum(-1)).mean())
assert abs(analytic - mc) < 0.02 + 0.02 * abs(mc), f"analytic {analytic:.4f} vs sampled {mc:.4f}"

# 4. model_kl adds up over layers.
two = nn.Sequential(BayesLinear(3, 4), nn.ReLU(), BayesLinear(4, 2))
expected = float(two[0].kl_divergence().detach()) + float(two[2].kl_divergence().detach())
assert abs(float(model_kl(two).detach()) - expected) < 1e-6

print(f"OK   KL closed form {analytic:.4f} matches the sampled estimate {mc:.4f}")

## 4.3 The objective, correctly scaled

This is the step people get wrong, so it is worth being explicit. The ELBO is a
quantity for the whole dataset:

    L = sum over all N points of E_q[log p(y_i | x_i, W)]  -  KL(q || p)

The likelihood term is a sum over `N` points. The KL term appears **once**, not
once per point. But training runs on minibatches, and the provided loss functions
already return the *mean* over the batch. Divide the whole objective by `N` so the
units match:

    -L / N  =  mean over the batch of -log p(y | x, W)  +  KL(q || p) / N

So `elbo_loss` returns `base_loss(...) + model_kl(model) / n_train`. Not
`/ batch_size`, and not unscaled. `fit` passes `n_train` for exactly this.

Get it wrong and nothing crashes. Write the tempered objective
`NLL + beta * KL / N` and read off what each mistake means, for `N = 200` and a
batch of 128:

| what you wrote | beta | what you get |
|---|---|---|
| `/ n_train` | 1 | correct |
| `/ batch_size` | 1.6 | prior too strong |
| nothing | 200 | posterior collapses onto the prior: flat predictions, huge error bars |
| no KL term | 0 | an expensive deterministic network, near-zero epistemic uncertainty |

Both failure modes look like "it trained".

`n_mc` averages the likelihood term over several weight draws. One is usually
enough, because every minibatch draws fresh weights anyway.

In [ ]:
def elbo_loss(base_loss, n_mc=1):
    """Turn a per-example NLL into the negative ELBO, scaled per data point."""

    def loss(model, xb, yb, n_train):
        # ---- TODO --------------------------------------------------------
        # average base_loss over n_mc weight draws, then add the KL over the
        # whole model divided by n_train
        raise NotImplementedError
        # ------------------------------------------------------------------

    return loss

In [ ]:
# ---- check your work: the scaling ------------------------------------------
torch.manual_seed(5)

# Comparing the same batch at two declared dataset sizes isolates the scaling:
# the likelihood term is unchanged, so the difference must be exactly the
# difference of the two KL weights.
model_s = nn.Sequential(BayesLinear(2, 1))
set_sampling(model_s, False)  # remove sampling noise so the check is exact


def squared_error(m, xb, yb, n_train):
    return ((m(xb) - yb) ** 2).mean()


loss_fn = elbo_loss(squared_error)
xb, yb = torch.randn(16, 2), torch.randn(16, 1)
l100 = float(loss_fn(model_s, xb, yb, 100).detach())
l1000 = float(loss_fn(model_s, xb, yb, 1000).detach())
kl = float(model_kl(model_s).detach())
expected = kl / 100 - kl / 1000
assert abs((l100 - l1000) - expected) < 1e-4 * max(1.0, abs(expected)), (
    f"KL is not scaled by 1/n_train: got {l100 - l1000:.6f}, expected {expected:.6f}"
)

# With no Bayesian layer there is no KL, so the ELBO loss is the base loss.
plain = nn.Sequential(nn.Linear(2, 1))
plain_elbo = float(elbo_loss(squared_error)(plain, xb, yb, 50).detach())
assert abs(plain_elbo - float(squared_error(plain, xb, yb, 50).detach())) < 1e-6

print("OK   the KL is scaled by 1/n_train")

## 4.4 Training, and predicting

Two things differ from the earlier notebooks.

`weight_decay=0`. Weight decay is a Gaussian prior on the weights, and your KL
term already contains one. Leaving the default in place applies the prior twice,
with a strength that depends on the dataset size.

More epochs and a smaller learning rate. From the reparameterization gradient, the
signal reaching `log sigma` carries a factor of `sigma` and an `eps` that averages
to zero, so the widths converge much more slowly than the means. The sampled-weight
objective is also noisier, and `lr=1e-2` diverges.

In [ ]:
PRIOR_STD = 1.0
INIT_LOG_SIGMA = -5.0
S = 50  # weight draws at test time

set_seed(0)
model = build_model(
    ds,
    hidden=(64, 64),
    layer=partial(BayesLinear, prior_std=PRIOR_STD, init_log_sigma=INIT_LOG_SIGMA),
)
print(f"variational parameters: {count_parameters(model)} (two per weight)")

t0 = time.perf_counter()
history = fit(
    model,
    ds.x_train,
    ds.y_train,
    loss_fn=elbo_loss(default_loss(ds)),
    epochs=3000,
    lr=5e-3,
    weight_decay=0.0,  # the prior is already in the loss
    seed=0,
)
print(f"trained in {time.perf_counter() - t0:.1f}s, final negative ELBO {history[-1]:.4f}")

Now `predict_bbb`: make sure sampling is on, then run `S` forward passes. Each one
draws a different network from the fitted posterior, which is what the sample axis
means here.

In [ ]:
@torch.no_grad()
def predict_bbb(model, x, n_samples=50):
    """S forward passes, each with a fresh draw of the weights. Returns [S, N] each."""
    # ---- TODO ------------------------------------------------------------
    # almost the same as predict_mc_dropout: eval mode, set_sampling(model, True),
    # then n_samples passes through gaussian_head(model(x)), stacked
    raise NotImplementedError
    # ----------------------------------------------------------------------

## 4.5 The check that matters

A one-layer Bayesian network with no activation *is* Bayesian linear regression,
and there the posterior is known exactly:

    Sigma = (X^T X / noise^2 + I / prior^2)^-1
    mu    = Sigma X^T y / noise^2

So point your machinery at that problem and compare. If this passes, your ELBO,
your KL and your scaling are all correct together, which is much stronger evidence
than each being individually plausible.

This cell is provided; you only run it. It takes about 20 seconds. Mean-field VI
gets the posterior mean essentially exactly; what it gets wrong is the covariance,
which is the subject of the bonus notebook.

In [ ]:
# ---- check your work: against an exactly known posterior -------------------
torch.manual_seed(0)
n, d = 400, 3
noise_std, prior_std = 0.3, 1.0
x_blr = torch.randn(n, d)
w_true = torch.tensor([1.5, -2.0, 0.5])
y_blr = (x_blr @ w_true + noise_std * torch.randn(n)).reshape(-1, 1)

precision = x_blr.T @ x_blr / noise_std**2 + torch.eye(d) / prior_std**2
exact_cov = torch.linalg.inv(precision)
exact_mean = exact_cov @ x_blr.T @ y_blr.reshape(-1) / noise_std**2

blr = nn.Sequential(BayesLinear(d, 1, prior_std=prior_std, init_log_sigma=-3.0))
with torch.no_grad():
    blr[0].bias_mu.zero_()
    blr[0].bias_log_sigma.fill_(-20.0)  # pin the bias so the model matches BLR
blr[0].bias_mu.requires_grad_(False)
blr[0].bias_log_sigma.requires_grad_(False)


def fixed_noise_nll(m, xb, yb, n_train):
    return (0.5 * ((yb - m(xb)) / noise_std) ** 2).mean()


fit(
    blr,
    x_blr,
    y_blr,
    loss_fn=elbo_loss(fixed_noise_nll),
    epochs=3000,      # the widths are the last thing to converge
    batch_size=n,     # full batch, so there is no minibatch noise in the check
    lr=2e-2,
    weight_decay=0.0,
    seed=0,
)

vi_mean = blr[0].weight_mu.detach().flatten()
vi_std = torch.exp(blr[0].weight_log_sigma.detach()).flatten()
exact_std = torch.sqrt(torch.diag(exact_cov))

print("posterior mean, VI   ", [f"{v:+.3f}" for v in vi_mean.tolist()])
print("posterior mean, exact", [f"{v:+.3f}" for v in exact_mean.tolist()])
print("posterior std,  VI   ", [f"{v:.4f}" for v in vi_std.tolist()])
print("posterior std,  exact", [f"{v:.4f}" for v in exact_std.tolist()])

assert torch.allclose(vi_mean, exact_mean, atol=0.05), "the posterior mean is wrong"
assert torch.allclose(vi_std, exact_std, rtol=0.25), "the posterior widths are wrong"
print("\nOK   variational inference recovered a posterior known in closed form")

## 4.6 What it looks like on toy1d

In [ ]:
mu_g, sd_g = predict_bbb(model, grid, n_samples=S)
tot_g, ale_g, epi_g = decompose_variance(mu_g, sd_g)

fig, ax = plt.subplots(figsize=(9, 5))
plot_band(ax, grid, mu_g.mean(0), ale_g.sqrt(), tot_g.sqrt(), ds=ds, title=f"Bayes by Backprop, S={S}")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()

In [ ]:
plot_uncertainty_vs_x({"bbb": epi_g.sqrt()}, grid, ds);

## 4.7 Where the noise estimate goes wrong

`toy1d` is the one dataset where the true observation noise is known, so the
aleatoric estimate can be checked rather than trusted. Compare the estimated
`sigma(x)` with the truth.

In [ ]:
curves = {
    "deterministic": load_run("toy1d", "deterministic")["arrays"]["sd_aleatoric"],
    "mc_dropout": load_run("toy1d", "mc_dropout")["arrays"]["sd_aleatoric"],
    "bbb": ale_g.sqrt(),
}
plot_aleatoric_recovery(curves, grid)

# Averaged over the training range only: beyond it there is no "true" noise to
# compare against, and every method's sigma runs away.
gx = grid.numpy().ravel()
inside = np.abs(gx) <= 3.0
print(f"mean noise std over -3 < x < 3   true {float(np.mean(toy1d_noise_std(gx[inside]))):.3f}")
for name, c in curves.items():
    print(f"{name:34} {float(np.mean(np.asarray(c).ravel()[inside])):.3f}")

The Bayes by Backprop estimate is far too large, and the reason is worth working
through because it applies to every method here.

Hold the weights random and ask what value of `sigma^2` minimises the expected
negative log-likelihood. Writing `f(x, W)` for the sampled mean output,

    E_q[(y - f)^2] = (y - mean_q f)^2 + Var_q[f]

so the best `sigma^2` is the true noise, plus the model's squared mis-fit, plus
the variance the weight noise itself injects into the output. Under weight noise
the network cannot fit precisely, so the residuals are large, and the
heteroscedastic head books the difference as observation noise. Part of what this
model reports as aleatoric is epistemic.

The consequence for your report: the *total* predictive variance can be about
right, which is why the calibration numbers below are good, while the *split*
between the two terms is not trustworthy. Do not quote this aleatoric estimate as
a measurement of the noise in your data.

## 4.8 Scored, next to the other three

In [ ]:
mu_id, sd_id = predict_bbb(model, ds.x_test, n_samples=S)
mu_ood, sd_ood = predict_bbb(model, ds.x_ood, n_samples=S)

metrics = evaluate(
    (mu_id, sd_id),
    ds.y_test,
    (mu_ood, sd_ood),
    ds.y_ood,
    task=ds.task,
    decompose=decompose_variance,
    calibration=calibration_error,
)
metrics["train_s"] = 0.0

previous = {n: load_run("toy1d", n)["metrics"] for n in ("deterministic", "mc_dropout", "ensemble")}
print(results_table(previous | {"bbb": metrics}))

Expect the worst RMSE of the four. The likelihood term is evaluated under weight
noise, so a sharp fit costs KL, and the objective trades accuracy for staying near
the prior. Whether that trade pays off depends on what you need the model for, and
the comparison notebook is where you decide.

## 4.9 Your own track

One extra setting here. `track_hparams` may give an `init_log_sigma` different
from `-5`: Track A's target is sharply determined, its residual scatter is a few
percent of the target's spread, and at `exp(-5)` the initial weight noise is as
large as the whole signal, so the network never fits and reports an observation
noise larger than the range of the data. Starting from `exp(-7)` lets the means
fit first and the widths grow afterwards.

In [ ]:
TRACK = "A"  # <-- keep the same track for the whole project

ds_track = load_track(TRACK)
hp = track_hparams(TRACK, "bbb")
print(ds_track)
print("hyperparameters:", hp)

t0 = time.perf_counter()
set_seed(0)
model_track = build_model(
    ds_track,
    hidden=(64, 64),
    layer=partial(
        BayesLinear,
        prior_std=PRIOR_STD,
        init_log_sigma=hp.get("init_log_sigma", INIT_LOG_SIGMA),
    ),
)
fit(
    model_track,
    ds_track.x_train,
    ds_track.y_train,
    loss_fn=elbo_loss(default_loss(ds_track)),
    epochs=int(hp["epochs"]),
    lr=hp["lr"],
    weight_decay=0.0,
    seed=0,
)
train_s = time.perf_counter() - t0
print(f"trained in {train_s:.1f}s")

In [ ]:
@torch.no_grad()
def predict_probs_bbb(model, x, n_samples=50):
    """The classification version: S sampled softmax vectors, [S, N, K]."""
    model.eval()
    set_sampling(model, True)
    return torch.stack([torch.softmax(model(x), dim=-1) for _ in range(n_samples)])


if ds_track.task == "regression":
    row = evaluate(
        predict_bbb(model_track, ds_track.x_test, n_samples=S),
        ds_track.y_test,
        predict_bbb(model_track, ds_track.x_ood, n_samples=S),
        ds_track.y_ood,
        task=ds_track.task,
        decompose=decompose_variance,
        calibration=calibration_error,
    )
else:
    row = evaluate(
        (predict_probs_bbb(model_track, ds_track.x_test, n_samples=S),),
        ds_track.y_test,
        (predict_probs_bbb(model_track, ds_track.x_ood, n_samples=S),),
        None,
        task=ds_track.task,
        decompose=decompose_entropy,
        calibration=calibration_error_probs,
    )

row["train_s"] = round(train_s, 2)
prev_track = {n: load_run(TRACK, n)["metrics"] for n in ("deterministic", "mc_dropout", "ensemble")}
print(results_table(prev_track | {"bbb": row}))
save_run(TRACK, "bbb", row)

In [ ]:
levels = np.linspace(0.05, 0.95, 12)
coverage = [interval_coverage(mu_id, sd_id, ds.y_test, float(q)) for q in levels]
plot_reliability({"bbb": (levels, coverage)});

save_run(
    "toy1d",
    "bbb",
    metrics,
    mean=mu_g.mean(0),
    sd_aleatoric=ale_g.sqrt(),
    sd_total=tot_g.sqrt(),
    epistemic_std=epi_g.sqrt(),
    levels=levels,
    coverage=np.array(coverage),
)

## Done when

* all four check cells print `OK`, including the Bayesian linear regression one;
* the band widens in the gap and the aleatoric estimate is visibly too high;
* `results/toy1d/bbb.json` and `results/<your track>/bbb.json` exist.

Next: notebook 05 puts the four methods in one table and asks what the numbers
mean.